In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Consistent aesthetics for seaborn/matplotlib
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

print("Improting done")

✓ All imports loaded


In [2]:
df = pd.read_csv('../data/raw/ai4i2020.csv')


print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(5)

Shape: (10000, 14)
Columns: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [23]:
# ── CELL 3: Sanity checks ───────────────────────────

# 3a. Data types
print("=== DATA TYPES ===")
print(df.dtypes)

# 3b. Missing values — expect zero for this dataset
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

# 3c. Duplicate rows
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# 3d. Summary statistics
df.describe().round(2)

# ── CELL 3e: Class balance check ────────────────────
failure_rate = df['Machine failure'].mean() * 100
print(f"Overall failure rate: {failure_rate:.1f}%")
print(df['Machine failure'].value_counts())

# Failure breakdown by type
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
print("\n=== FAILURES BY TYPE ===")
print(df[failure_cols].sum().sort_values(ascending=False))

# ── CELL 3f: Product quality distribution ───────────
print("Product quality distribution:")
print(df['Type'].value_counts())
print("\nFailure rate by product type:")
print(df.groupby('Type')['Machine failure'].mean().mul(100).round(1))

=== DATA TYPES ===
UDI                          int64
Product ID                     str
Type                           str
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

=== MISSING VALUES ===
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

Duplicate rows: 0
Overall failure

In [28]:
# ── CELL 4: Data Cleaning ───────────────────────────

# 4a. Copy raw — never modify df directly
df_clean = df.copy()

# 4b. Rename columns to be Python-friendly
df_clean.columns = [
    'udi', 'product_id', 'type',
    'air_temp_k', 'process_temp_k', 'rpm',
    'torque_nm', 'tool_wear_min', 'machine_failure',
    'twf', 'hdf', 'pwf', 'osf', 'rnf'
]

# 4c. Drop identifier columns (not useful for analysis)
df_clean = df_clean.drop(columns=['udi', 'product_id'])

# 4d. Encode product type: L=0, M=1, H=2
type_map = {'L': 0, 'M': 1, 'H': 2}
df_clean['type_encoded'] = df_clean['type'].map(type_map)

# 4e. Drop RNF column — it is random noise, not a real failure mode
# (documented in the original paper: RNF = random noise failures)
df_clean = df_clean.drop(columns=['rnf'])

print(f"Clean dataset shape: {df_clean.shape}")
print("Columns kept:", list(df_clean.columns))
df_clean.head(3)

Clean dataset shape: (10000, 12)
Columns kept: ['type', 'air_temp_k', 'process_temp_k', 'rpm', 'torque_nm', 'tool_wear_min', 'machine_failure', 'twf', 'hdf', 'pwf', 'osf', 'type_encoded']


,type,air_temp_k,process_temp_k,rpm,torque_nm,tool_wear_min,machine_failure,twf,hdf,pwf,osf,type_encoded
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,1
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0


In [29]:
df_clean.to_csv('../data/processed/manufacturing_clean.csv', index=False)
print("Saved to data/processed/manufacturing_clean.csv")

Saved to data/processed/manufacturing_clean.csv


In [30]:
# ── CELL 6: Feature Engineering ─────────────────────

# 6a. Thermal differential (process heat minus ambient)
df_clean['temp_diff'] = df_clean['process_temp_k'] - df_clean['air_temp_k']

# 6b. Mechanical power (convert rpm to rad/s first)
df_clean['power_w'] = df_clean['torque_nm'] * (df_clean['rpm'] * 2 * np.pi / 60)

# 6c. Tool wear bins (5 equal-width stages)
df_clean['wear_bin'] = pd.cut(
    df_clean['tool_wear_min'],
    bins=[0, 50, 100, 150, 200, 260],
    labels=['Fresh', 'Light', 'Moderate', 'Heavy', 'Critical']
)

# 6d. Overstrain proxy (high wear + high torque = compounded stress)
df_clean['overstrain'] = df_clean['tool_wear_min'] * df_clean['torque_nm']

# Verify new columns
print("New features added:")
print(df_clean[['temp_diff', 'power_w', 'wear_bin', 'overstrain']].describe().round(1))

# 6e. Re-save the enriched dataset
df_clean.to_csv('../data/processed/manufacturing_features.csv', index=False)
print("✓ Feature-engineered dataset saved")

New features added:
       temp_diff  power_w  overstrain
count    10000.0  10000.0     10000.0
mean        10.0   6279.7      4314.7
std          1.0   1067.4      2826.6
min          7.6   1148.4         0.0
25%          9.3   5561.2      1963.6
50%          9.8   6271.0      4013.0
75%         11.0   7003.0      6279.0
max         12.1  10469.9     16497.0
✓ Feature-engineered dataset saved
